
Answers multi-hop questions over 609 news articles (`corpus.json`). Every question needs facts from 2–4 articles, usually from named outlets. The pipeline returns the answer, the question type and the supporting evidence, or **"Insufficient information."** when the corpus can't support an answer.

```
OFFLINE (once)  corpus → clean → sentences → semantic chunks (≤ 200 tokens, title prefixed)
                → GPT4All embeddings (384 numbers, normalised) → LangChain FAISS store (flat = exact)

PER QUESTION    1 PLAN       Gemini call #1: question type + facts, each with outlet/date   (temperature 0)
                2 RETRIEVE   per fact: metadata filter first → cosine ranking → top 3        (no LLM)
                3 CHECK      weak fact → loosen filters (no LLM) → one REPAIR call (#2)
                             → still weak → "Insufficient information."                      (no answer call)
                4 ANSWER     Gemini call #3: labelled passages + type-specific instruction
                5 CONFIDENCE logistic regression on measured signals → abstain below threshold
```

**Evaluation** uses the labels in `train.json` (answer, question type, gold evidence articles), split by question type into a **validation** part (choose thresholds) and a **test** part (report numbers once):

| Claim | Metric in this notebook |
|---|---|
| Fewer question-type misclassifications | Error rate: bare prompt vs the planner prompt, McNemar test |
| Query relevance | Precision@3 of retrieved chunks against the gold evidence articles |
| Accuracy with confidence gating | Accuracy on answered questions, reported with coverage |
| Fewer hallucinations | False-answer rate on unanswerable questions, with vs without gating |

**How to run**
1. Put `corpus.json` and `train.json` (from the repo's `SOTA_RAG` folder) next to this notebook, or set `DATA_DIR`.
2. Set the `GEMINI_API_KEY` environment variable.
3. Run all cells. `SAMPLE_PER_TYPE` keeps the first run small; set it to `None` for the full 2,556 questions.
4. `DRY_RUN = True` runs everything with stand-in models (no API key, no downloads). It only checks the plumbing; its numbers mean nothing.


In [ ]:
# Fresh environment / Colab: install dependencies
!pip -q install google-generativeai gpt4all faiss-cpu langchain-community langchain-core nltk python-dateutil scikit-learn transformers

In [ ]:
import os, re, json, time, math, random, hashlib, difflib
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from dateutil import parser as date_parser
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
from scipy.stats import binomtest

from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

## 1. Configuration

In [ ]:
DATA_DIR = "."                         # folder with corpus.json and train.json
WORK_DIR = "factflow_artifacts"        # index, LLM cache and outputs go here

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
PLANNER_MODEL = "gemini-1.0-pro"       # planning and repair: structured extraction, a small model is enough
ANSWER_MODEL  = "gemini-1.0-pro"       # writes the answer from the passages
JUDGE_MODEL   = "gemini-1.0-pro"       # only for ambiguous name matches during evaluation
LLM_CALLS_PER_MINUTE = 50              # stay under your API quota

MAX_CHUNK_TOKENS = 200                 # chunk cap, counted with the embedding model's tokenizer
CUT_PERCENTILE = 15                    # semantic chunking: cut at the lowest 15% neighbour similarities
TOP_K = 3                              # chunks kept per fact
MAX_FACTS = 4                          # the data has 2-4 evidence articles per question

VAL_FRACTION = 0.30                    # validation share of each question type
SAMPLE_PER_TYPE = 100                  # questions per type for a first run; None = all 2,556
COST_OF_WRONG = 2.0                    # when choosing the confidence threshold: a wrong answer costs 2x a right one
USE_LLM_JUDGE = True                   # LLM judge only for name answers that code can't match
USE_UNKNOWN_OUTLET_EXIT = True         # dataset-specific: an outlet the corpus lacks → abstain

DRY_RUN = False                        # True = stand-in models, for testing the code path only
SEED = 42

MODE = "dry" if DRY_RUN else "gpt4all"
os.makedirs(WORK_DIR, exist_ok=True)
INDEX_DIR  = os.path.join(WORK_DIR, f"faiss_store_{MODE}")
CACHE_PATH = os.path.join(WORK_DIR, f"llm_cache_{MODE}.json")
random.seed(SEED); np.random.seed(SEED)
ABSTAIN = "Insufficient information."

## 2. Data and the validation / test split

The three questions used as worked examples in the planner prompt are removed from both splits, so the examples never leak into evaluation.

In [ ]:
corpus = json.load(open(os.path.join(DATA_DIR, "corpus.json")))
questions = json.load(open(os.path.join(DATA_DIR, "train.json")))
for i, q in enumerate(questions):
    q["qid"] = i

EXAMPLE_MARKERS = ["welcome bonus of up to $1000 in bonus bets",
                   "Australia's Davis Cup team is aiming",
                   "Jada Pinkett Smith's statements regarding what she learned"]
example_qids = {q["qid"] for q in questions if any(m in q["query"] for m in EXAMPLE_MARKERS)}

by_type = defaultdict(list)
for q in questions:
    if q["qid"] not in example_qids:
        by_type[q["question_type"]].append(q)

rng = random.Random(SEED)
val_qs, test_qs = [], []
for qtype in sorted(by_type):
    qs = by_type[qtype][:]
    rng.shuffle(qs)
    if SAMPLE_PER_TYPE:
        qs = qs[:SAMPLE_PER_TYPE]
    n_val = round(len(qs) * VAL_FRACTION)
    val_qs += qs[:n_val]
    test_qs += qs[n_val:]

print(f"corpus: {len(corpus)} articles | questions: {len(questions)} (examples held out: {len(example_qids)})")
print(pd.DataFrame({"validation": Counter(q["question_type"] for q in val_qs),
                    "test": Counter(q["question_type"] for q in test_qs)}))

## 3. The Gemini call: cache, rate limit, retries

Every call is cached on disk, keyed by model + settings + prompt, so re-running the notebook doesn't pay twice. All calls use **temperature 0**: planning, answering and judging are extraction tasks where we want the most likely output, consistently.

In [ ]:
LLM_CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
CALLS = Counter()                      # real API calls made, by purpose

def save_cache():
    with open(CACHE_PATH, "w") as f:
        json.dump(LLM_CACHE, f)

if not DRY_RUN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)

_last_call = [0.0]
def wait_for_rate_limit():
    gap = 60.0 / LLM_CALLS_PER_MINUTE
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

PERMANENT_ERRORS = ("not found", "api key", "permission", "invalid argument", "unsupported")

def call_llm(prompt, model, purpose, temperature=0.0, max_tokens=512):
    key = hashlib.sha256(f"{model}|{temperature}|{max_tokens}|{prompt}".encode()).hexdigest()
    if key in LLM_CACHE:
        return LLM_CACHE[key]
    CALLS[purpose] += 1
    if DRY_RUN:
        text = fake_llm(prompt)
    else:
        text = ""
        for attempt in range(5):
            wait_for_rate_limit()
            try:
                response = genai.GenerativeModel(model).generate_content(
                    prompt,
                    generation_config=genai.types.GenerationConfig(temperature=temperature,
                                                                   max_output_tokens=max_tokens))
                try:
                    text = response.text
                except ValueError:             # blocked by safety filters or no text returned
                    text = ""
                break
            except Exception as e:
                if any(s in str(e).lower() for s in PERMANENT_ERRORS) or attempt == 4:
                    raise                      # don't retry errors that will never succeed
                time.sleep(2 ** attempt + random.random())   # backoff + jitter for 429 / 5xx
    LLM_CACHE[key] = text
    if len(LLM_CACHE) % 25 == 0:
        save_cache()
    return text

def fill(template, **values):
    # replace <<name>> placeholders; avoids escaping the JSON braces in prompts
    for k, v in values.items():
        template = template.replace(f"<<{k}>>", str(v))
    return template

def extract_json(text):
    # Gemini 1.0 had no JSON mode: take the first {...} block from the reply
    text = re.sub(r"```(?:json)?", "", text or "")
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None

def fake_llm(prompt):
    # DRY_RUN stand-in. Crude rules, only so every code path runs without an API key.
    question = prompt.split("Question:")[-1].strip().split("\n")[0]
    if prompt.startswith(("TASK: PLAN", "TASK: REPAIR")):
        named = [s for s in STORED_SOURCES if simplify_source(s) and simplify_source(s) in re.sub(r"[^a-z0-9]", "", question.lower())]
        facts = [{"text": question[:160], "source": s, "date": None} for s in named[:MAX_FACTS]]
        return json.dumps({"question_type": rule_based_type(question),
                           "facts": facts or [{"text": question[:160], "source": None, "date": None}]})
    if prompt.startswith("TASK: CLASSIFY"):
        return rule_based_type(question).replace("_query", "")
    if prompt.startswith("TASK: ANSWER"):
        if "Reply Yes or No" in prompt:
            return random.choice(["Yes", "No"])
        names = re.findall(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", prompt.split("Passages:")[-1])
        return names[0] if names else ABSTAIN
    if prompt.startswith("TASK: JUDGE"):
        return "DIFFERENT"
    return ""

## 4. Embeddings and tokenizer

GPT4All's default embedding model (all-MiniLM-L6-v2) gives **384 numbers per text**. Every vector is normalised to length 1, so a dot product equals cosine similarity. Chunks and questions **must** use this same model: vectors from different models live in different spaces and can't be compared.

The tokenizer is used only to **count** tokens, so each chunk fits the embedding model's input limit. Chunks themselves stay as text.

In [ ]:
EMBED_DIM = 384

if DRY_RUN:
    def embed_raw(texts):              # hashing stand-in: similar words → similar vectors
        out = np.zeros((len(texts), EMBED_DIM), dtype="float32")
        for i, t in enumerate(texts):
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                h = int(hashlib.md5(w.encode()).hexdigest(), 16)
                out[i, h % EMBED_DIM] += 1.0 if (h >> 9) & 1 else -1.0
        return out
else:
    from gpt4all import Embed4All
    gpt4all_embedder = Embed4All()     # downloads the small embedding model on first use
    def embed_raw(texts):
        return np.array(gpt4all_embedder.embed(list(texts)), dtype="float32")

def embed_texts(texts, batch_size=128):
    parts = [embed_raw(texts[i:i + batch_size]) for i in range(0, len(texts), batch_size)]
    vecs = np.vstack(parts) if parts else np.zeros((0, EMBED_DIM), dtype="float32")
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (vecs / norms).astype("float32")

class GPT4AllEmbeddingsAdapter(Embeddings):
    # LangChain interface around embed_texts, so the store embeds queries exactly like chunks
    def embed_documents(self, texts):
        return embed_texts(list(texts)).tolist()
    def embed_query(self, text):
        return embed_texts([text])[0].tolist()

EMBEDDINGS = GPT4AllEmbeddingsAdapter()

try:
    if DRY_RUN:
        raise ImportError
    from transformers import AutoTokenizer
    _tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    def count_tokens(text):
        return len(_tokenizer.encode(text, add_special_tokens=False))
except Exception:
    def count_tokens(text):            # fallback estimate: ~1.3 tokens per word
        return int(len(text.split()) * 1.3) + 1

## 5. Cleaning and semantic chunking

1. **Clean:** drop boilerplate lines ("Table of Contents", ads, sign-up prompts).
2. **Sentences:** a real sentence splitter, so "Dr.", "$1.5" and "U.S." don't break sentences.
3. **Semantic cuts:** embed every sentence and cut where neighbouring sentences are least similar (topic change), so each chunk is about one thing.
4. **Size cap:** pack sentences so each chunk, plus its title prefix, stays within `MAX_CHUNK_TOKENS`.
5. Each finished chunk is **re-embedded as one text** when the index is built (not an average of its sentence vectors), so links across sentences are captured.

In [ ]:
JUNK_LINE = re.compile(r"^(table of contents|advertisement|sign up|subscribe|read more|related:|share this|click here)", re.I)

def clean(text):
    lines = []
    for line in (text or "").split("\n"):
        line = re.sub(r"\s+", " ", line).strip()
        if line and not JUNK_LINE.match(line):
            lines.append(line)
    return "\n".join(lines)

try:
    if DRY_RUN:
        raise ImportError
    import nltk
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    from nltk.tokenize import sent_tokenize
    sent_tokenize("Test one. Test two.")
except Exception:
    def sent_tokenize(text):           # fallback: split after . ! ? followed by a capital letter
        return re.split(r"(?<=[.!?])\s+(?=[A-Z\"'])", text)

def split_sentences(text):
    sentences = []
    for paragraph in text.split("\n"):  # never join sentences across paragraphs
        sentences += [s.strip() for s in sent_tokenize(paragraph) if s.strip()]
    return sentences

def split_long(sentence, max_tokens):
    # rare: a single "sentence" longer than the cap (lists, run-on text) is cut by words
    if count_tokens(sentence) <= max_tokens:
        return [sentence]
    words, step = sentence.split(), max(10, int(max_tokens / 1.5))
    return [" ".join(words[i:i + step]) for i in range(0, len(words), step)]

def pack_sentences(sentences, max_tokens):
    # greedy: keep adding whole sentences while the chunk fits
    chunks, current, current_tokens = [], [], 0
    for s in sentences:
        n = count_tokens(s) + 1
        if current and current_tokens + n > max_tokens:
            chunks.append(" ".join(current))
            current, current_tokens = [], 0
        current.append(s)
        current_tokens += n
    if current:
        chunks.append(" ".join(current))
    return chunks

def semantic_chunks(sentences, sentence_vecs, max_tokens):
    if len(sentences) <= 2:
        return pack_sentences(sentences, max_tokens)
    sims = (sentence_vecs[:-1] * sentence_vecs[1:]).sum(axis=1)       # cosine of neighbours
    threshold = np.percentile(sims, CUT_PERCENTILE)                     # "low" relative to this article
    groups = [[sentences[0]]]
    for i, sim in enumerate(sims):
        if sim <= threshold:
            groups.append([])                                           # topic changed → new chunk
        groups[-1].append(sentences[i + 1])
    chunks = []
    for g in groups:
        chunks += pack_sentences(g, max_tokens)                         # enforce the size cap
    merged = []
    for c in chunks:                                                    # fold tiny chunks into the previous one
        if merged and count_tokens(c) < 25 and count_tokens(merged[-1] + " " + c) <= max_tokens:
            merged[-1] += " " + c
        else:
            merged.append(c)
    return merged

## 6. Build (or load) the FAISS store

Each chunk is stored with its metadata (ids, outlet, date, title, url), created in the same loop. `FAISS.from_embeddings` with inner-product distance builds a **flat** index: exact brute-force search, which takes about a millisecond at this size. No HNSW or IVF needed.

FAISS itself keeps only vectors, identified by position. LangChain keeps the text and metadata alongside them, and `save_local` writes both together.

In [ ]:
def simplify_source(name):
    # "The Independent - Life and Style" → "independent", "CBSSports.com" → "cbssportscom"
    name = str(name).lower().split(" | ")[0].split(" - ")[0]
    name = re.sub(r"^the ", "", name.strip())
    return re.sub(r"[^a-z0-9]", "", name)

def build_store():
    texts, metadatas = [], []
    for article_id, art in enumerate(corpus):
        title = (art.get("title") or "").strip()
        budget = max(60, MAX_CHUNK_TOKENS - count_tokens(title) - 2)    # leave room for the title prefix
        sentences = []
        for s in split_sentences(clean(art.get("body"))):
            sentences += split_long(s, budget)
        if not sentences:
            continue
        sentence_vecs = embed_texts(sentences)                          # only used to find topic changes
        for chunk_index, chunk in enumerate(semantic_chunks(sentences, sentence_vecs, budget)):
            published = pd.Timestamp(art["published_at"])
            texts.append(f"{title} — {chunk}")
            metadatas.append({
                "article_id": article_id, "chunk_index": chunk_index, "title": title,
                "url": art.get("url") or "", "source": art.get("source") or "",
                "source_key": simplify_source(art.get("source") or ""),
                "published_at": art["published_at"], "published_ts": int(published.timestamp()),
                "category": art.get("category") or "",
            })
        if article_id % 100 == 0:
            print(f"  chunked {article_id}/{len(corpus)} articles, {len(texts)} chunks so far")
    vectors = embed_texts(texts)                                         # re-embed each whole chunk
    store = FAISS.from_embeddings(list(zip(texts, vectors.tolist())), EMBEDDINGS, metadatas=metadatas,
                                  distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)
    store.save_local(INDEX_DIR)
    return store

if os.path.exists(INDEX_DIR):
    store = FAISS.load_local(INDEX_DIR, EMBEDDINGS, allow_dangerous_deserialization=True,   # our own file
                             distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)
else:
    t0 = time.time()
    store = build_store()
    print(f"built in {time.time() - t0:.0f}s")

N_CHUNKS = store.index.ntotal
VECS = store.index.reconstruct_n(0, N_CHUNKS)                            # row i = vector at FAISS position i
_docs = [store.docstore.search(store.index_to_docstore_id[i]) for i in range(N_CHUNKS)]
META = pd.DataFrame([dict(d.metadata, text=d.page_content) for d in _docs])
assert len(META) == N_CHUNKS == len(VECS), "vectors and metadata are out of line"

tok = META["text"].map(count_tokens)
print(f"{N_CHUNKS} chunks from {META['article_id'].nunique()} articles | tokens per chunk: "
      f"median {tok.median():.0f}, max {tok.max()}")

## 7. Outlet names and dates: the LLM writes them, code makes them exact

The planner writes outlet names and dates the way the question does. Code converts them to the values stored in the metadata, otherwise an exact filter would silently match nothing. Matching is exact name, then simplified name, then a very close fuzzy match. If none matches, the outlet is treated as **not in the corpus**.

In [ ]:
STORED_SOURCES = sorted(META["source"].unique())
KNOWN_KEYS = set(META["source_key"].unique())

def to_source_key(written):
    # returns (source_key or None, recognised?)
    if written is None or str(written).strip().lower() in ("", "null", "none"):
        return None, True
    key = simplify_source(written)
    if key in KNOWN_KEYS:
        return key, True
    close = difflib.get_close_matches(key, KNOWN_KEYS, n=1, cutoff=0.85)
    if close:
        return close[0], True
    return None, False

def to_date(text):
    if not text or str(text).strip().lower() in ("null", "none"):
        return None
    try:
        return pd.Timestamp(date_parser.parse(str(text), default=pd.Timestamp("2023-01-01")))
    except (ValueError, OverflowError):
        return None

print(to_source_key("The Independent"), to_source_key("Sporting News"), to_source_key("Rolling Stone"))
print(to_date("October 7, 2023"), to_date("2023-10-30"), to_date("unknown"))

## 8. Step 1: Plan (Gemini call #1)

One small-model call turns the question into a **question type** plus a list of **facts**: one short search phrase per piece of information, with its outlet and date when the question gives them. Searching per fact avoids the main failure of a single search: one vector for a multi-fact question matches only the dominant fact.

The prompt gives precise type definitions, a decision rule for the confusable temporal/comparison pair, the outlet list and three worked examples. Code validates the reply and retries once with the problems listed; if it still fails, the whole question becomes a single fact, so the pipeline degrades gracefully.

In [ ]:
VALID_TYPES = ("inference_query", "comparison_query", "temporal_query")

def rule_based_type(question):
    q = question.strip()
    yes_no = re.match(r"(does|do|is|are|was|were|did|has|have|can|would|could)\b", q, re.I)
    if yes_no and re.search(r"\b(after|before|subsequent|earlier|later|following|prior)\b", q, re.I):
        return "temporal_query"
    return "comparison_query" if yes_no else "inference_query"

PLAN_EXAMPLES = '''Question: Which online betting platform provides a welcome bonus of up to $1000 in bonus bets for new customers' first losses, runs NBA betting promotions, and is anticipated to extend the same sign-up offer to new users in Vermont, as reported by both CBSSports.com and Sporting News?
{"question_type": "inference_query", "facts": [
  {"text": "welcome bonus up to $1000 in bonus bets if first bet loses", "source": "CBSSports.com", "date": null},
  {"text": "betting platform runs NBA betting promotions", "source": "Sporting News", "date": null},
  {"text": "same sign-up offer expected for new users in Vermont", "source": "Sporting News", "date": null}]}

Question: Does 'The Age' article suggest that Australia's Davis Cup team is aiming for an improvement in their performance compared to the previous year, while the 'Sporting News' article indicates that the South Africa national rugby team has already achieved an improvement to reach the Rugby World Cup semi-finals?
{"question_type": "comparison_query", "facts": [
  {"text": "Australia Davis Cup team aims to do better than last year", "source": "The Age", "date": null},
  {"text": "South Africa rugby team improved and reached the World Cup semi-finals", "source": "Sporting News", "date": null}]}

Question: Was there inconsistency in Jada Pinkett Smith's statements regarding what she learned from her children after The Independent - Life and Style's report on her views on self-acceptance published at 13:41:30, and before The Independent - Life and Style's subsequent report on the same topic published at 15:48:10 on the same day?
{"question_type": "temporal_query", "facts": [
  {"text": "Jada Pinkett Smith on what she learned from her children, self-acceptance (earlier report)", "source": "The Independent - Life and Style", "date": null},
  {"text": "Jada Pinkett Smith praises her children as her gurus (later report)", "source": "The Independent - Life and Style", "date": null}]}'''

PLANNER_PROMPT = '''TASK: PLAN
You plan searches over an archive of news articles from late 2023.

Question types:
- inference_query: asks WHO or WHAT (a person, company, product, team...) is described by several facts. The answer is a name.
- comparison_query: asks whether reports AGREE, DIFFER, or BOTH state something. The answer is Yes or No.
- temporal_query: asks whether something CHANGED or stayed CONSISTENT between reports published at DIFFERENT TIMES (words like after, before, subsequent, earlier, later). The answer is Yes or No.
Rule: a Yes/No question that refers to the order or timing of reports is temporal_query; any other Yes/No question is comparison_query.

Split the question into facts:
- one fact per separate piece of information the answer depends on (usually one per news report mentioned);
- write each fact as a short search phrase, not a question;
- "source": the news outlet that reported the fact, written as in the question, or null;
- "date": the report's date as YYYY-MM-DD if the question gives a date, otherwise null.
Known outlets (use this spelling when the question names one of them): <<sources>>

Examples:
<<examples>>

Reply with JSON only, no other text, in this format:
{"question_type": "...", "facts": [{"text": "...", "source": "... or null", "date": "YYYY-MM-DD or null"}]}

Question: <<question>>'''

def validate_plan(raw):
    if not isinstance(raw, dict):
        return None, ["the reply was not valid JSON"]
    problems = []
    qtype = raw.get("question_type")
    if qtype not in VALID_TYPES:
        problems.append(f"question_type must be one of {list(VALID_TYPES)}")
    facts = []
    for f in raw.get("facts") or []:
        if not isinstance(f, dict) or not str(f.get("text", "")).strip():
            continue
        written = f.get("source")
        written = None if written is None or str(written).strip().lower() in ("", "null", "none") else str(written).strip()
        key, recognised = to_source_key(written)
        facts.append({"id": f"f{len(facts) + 1}", "text": str(f["text"]).strip(),
                      "source_written": written, "source_key": key, "source_known": recognised,
                      "date": to_date(f.get("date"))})
    if not facts:
        problems.append("facts must be a non-empty list")
    return {"question_type": qtype, "facts": facts[:MAX_FACTS]}, problems

def whole_question_plan(question, qtype=None):
    return {"question_type": qtype if qtype in VALID_TYPES else rule_based_type(question),
            "facts": [{"id": "f1", "text": question, "source_written": None, "source_key": None,
                       "source_known": True, "date": None}]}

def plan_question(question):
    prompt = fill(PLANNER_PROMPT, sources=", ".join(STORED_SOURCES), examples=PLAN_EXAMPLES, question=question)
    plan, problems = validate_plan(extract_json(call_llm(prompt, PLANNER_MODEL, "plan")))
    if problems:
        retry = prompt + "\n\nYour previous reply had these problems: " + "; ".join(problems) + ". Reply again with valid JSON only."
        plan2, problems2 = validate_plan(extract_json(call_llm(retry, PLANNER_MODEL, "plan_retry")))
        if not problems2:
            plan, problems = plan2, []
    if plan is None or not plan["facts"]:
        return whole_question_plan(question, plan["question_type"] if plan else None)
    if plan["question_type"] not in VALID_TYPES:
        plan["question_type"] = rule_based_type(question)
    return plan

demo_plan = plan_question(test_qs[0]["query"])
print(test_qs[0]["query"], "\n")
for f in demo_plan["facts"]:
    print(" ", f["id"], "|", f["text"], "|", f["source_written"], "→", f["source_key"], "|", f["date"])
print("type:", demo_plan["question_type"], "| gold:", test_qs[0]["question_type"])

## 9. Step 2: Retrieve, per fact (no LLM)

All fact texts are embedded in one batched call. Each fact is then searched **filter-first**: metadata picks the allowed chunks (that outlet, that date window), and only those are ranked by cosine similarity. That guarantees the best matches **within** the named outlet. If the best score is weak, the filters are loosened step by step: outlet + date → outlet → everything. With no filter at all, the FAISS index searches every chunk.

In [ ]:
LEVELS = [(1, True, True),    # outlet + date window
          (2, True, False),   # outlet only
          (3, False, False)]  # no filter

SOURCE_KEYS = META["source_key"].values
PUB_TS = META["published_ts"].values

def allowed_positions(fact, use_source, use_date):
    mask = np.ones(N_CHUNKS, dtype=bool)
    if use_source:
        mask &= SOURCE_KEYS == fact["source_key"]
    if use_date:
        lo = (fact["date"] - pd.Timedelta(days=1)).timestamp()
        hi = (fact["date"] + pd.Timedelta(days=2)).timestamp()     # the given day ± 1
        mask &= (PUB_TS >= lo) & (PUB_TS < hi)
    return np.nonzero(mask)[0]

def search(qvec, positions, k):
    if positions is None:                                         # unfiltered: FAISS flat search
        scores, ids = store.index.search(qvec[None, :], k)
        return [(int(i), float(s)) for s, i in zip(scores[0], ids[0]) if i != -1]
    if len(positions) == 0:
        return []
    scores = VECS[positions] @ qvec                               # exact cosine on the allowed chunks
    top = np.argsort(-scores)[:k]
    return [(int(positions[t]), float(scores[t])) for t in top]

def hit_record(position, score):
    row = META.iloc[position]
    return {"position": position, "score": round(score, 4), "url": row["url"], "title": row["title"],
            "source": row["source"], "published_at": row["published_at"],
            "published_ts": int(row["published_ts"]), "text": row["text"]}

def retrieve_fact(fact, qvec, min_score, k=TOP_K):
    best = None
    for level, use_source, use_date in LEVELS:
        if (use_source and not fact["source_key"]) or (use_date and fact["date"] is None):
            continue                                              # nothing to filter on at this level
        positions = allowed_positions(fact, use_source, use_date) if (use_source or use_date) else None
        hits = [hit_record(p, s) for p, s in search(qvec, positions, k)]
        result = {"fact": fact, "level": level, "hits": hits, "best": hits[0]["score"] if hits else 0.0}
        if best is None or result["best"] > best["best"]:
            best = result
        if result["best"] >= min_score:
            return dict(result, strong=True)
    return dict(best, strong=False)

def retrieve_all(plan, min_score):
    qvecs = embed_texts([f["text"] for f in plan["facts"]])      # one batched embedding call
    return [retrieve_fact(f, v, min_score) for f, v in zip(plan["facts"], qvecs)]

for r in retrieve_all(demo_plan, min_score=0.5):
    print(r["fact"]["id"], "level", r["level"], "best", r["best"], "|", r["hits"][0]["source"] if r["hits"] else "-",
          "|", (r["hits"][0]["text"][:110] + "…") if r["hits"] else "")

## 10. Step 3: Check the evidence, repair once (Gemini call #2, only when needed)

If any fact is still weak after loosening the filters, **one** stateless repair call gets the question, the current plan, the failed facts and the evidence found for the other facts. It returns a corrected plan: it can rewrite a vague fact using names found in the other evidence (multi-hop chaining), fix an outlet or date, or re-split the question. If evidence is still missing after that, **code** answers "Insufficient information.". The model is never asked to answer without evidence.

In [ ]:
REPAIR_PROMPT = '''TASK: REPAIR
You planned searches for a news question, but some facts found no supporting article.

Question: <<question>>
Current plan: <<plan>>
Facts with NO evidence: <<failed>>
Evidence found for the other facts:
<<evidence>>

Return a corrected plan in the same JSON format. You may:
- rewrite a failed fact as a better search phrase, adding specific names found in the evidence above;
- correct a wrong source or date;
- re-split the question if the facts were divided wrongly.
Keep facts that already have evidence unchanged.
Known outlets: <<sources>>

Reply with JSON only:
{"question_type": "...", "facts": [{"text": "...", "source": "... or null", "date": "YYYY-MM-DD or null"}]}'''

def plan_as_json(plan):
    return json.dumps({"question_type": plan["question_type"],
                       "facts": [{"text": f["text"], "source": f["source_written"],
                                  "date": f["date"].strftime("%Y-%m-%d") if f["date"] is not None else None}
                                 for f in plan["facts"]]})

def repair_plan(question, plan, results):
    failed = [r["fact"]["text"] for r in results if not r["strong"]]
    found = [f'[{r["fact"]["id"]}] "{r["hits"][0]["text"][:400]}"' for r in results if r["strong"] and r["hits"]]
    prompt = fill(REPAIR_PROMPT, question=question, plan=plan_as_json(plan), failed=json.dumps(failed),
                  evidence="\n".join(found) or "(none)", sources=", ".join(STORED_SOURCES))
    new_plan, problems = validate_plan(extract_json(call_llm(prompt, PLANNER_MODEL, "repair")))
    if problems:
        return None
    if new_plan["question_type"] not in VALID_TYPES:
        new_plan["question_type"] = plan["question_type"]
    return new_plan

def n_strong(results):
    return sum(r["strong"] for r in results)

## 11. Step 4: Answer (Gemini call #3)

The prompt includes only what helps answering: each passage's text with its outlet, date and title. URLs, ids and scores stay in code. For temporal questions the passages are **sorted by publication time in code**, since the model is unreliable at comparing timestamps. The "Insufficient information" option is a second safety net: a high similarity score means the passage is on topic, not that it contains the answer.

In [ ]:
ANSWER_PROMPT = '''TASK: ANSWER
Answer the question using ONLY the passages below. Each passage shows its outlet and publication time.
If the passages do not contain enough information, reply exactly: Insufficient information.

Passages:
<<passages>>

Question: <<question>>
<<instruction>>'''

INSTRUCTIONS = {
    "inference_query":  "Reply with the name only (a person, organisation, team or thing), nothing else.",
    "comparison_query": "Check every part of the question against the passages. Reply Yes or No only.",
    "temporal_query":   "The passages are in publication order. Reply Yes or No only.",
}

def passages_for(results, qtype):
    seen, hits = set(), []
    for r in results:
        for h in r["hits"]:
            if h["position"] not in seen:
                seen.add(h["position"])
                hits.append(h)
    if qtype == "temporal_query":
        hits.sort(key=lambda h: h["published_ts"])
    blocks = []
    for n, h in enumerate(hits, 1):
        body = h["text"][len(h["title"]) + 3:] if h["text"].startswith(h["title"] + " — ") else h["text"]
        when = pd.Timestamp(h["published_at"]).strftime("%Y-%m-%d %H:%M UTC")
        blocks.append(f'[{n}] Outlet: {h["source"]} | Published: {when} | Title: {h["title"]}\n"{body}"')
    return "\n\n".join(blocks), " ".join(h["text"] for h in hits)

def parse_answer(text, qtype):
    line = (text or "").strip().strip('"').split("\n")[0].strip()
    line = re.sub(r"^(answer\s*:\s*)", "", line, flags=re.I).strip()
    if not line or "insufficient information" in line.lower():
        return ABSTAIN
    if qtype in ("comparison_query", "temporal_query"):
        words = re.findall(r"[a-z]+", line.lower())
        for w in words:
            if w in ("yes", "no"):
                return w.capitalize()
        return ABSTAIN                                    # not a usable Yes/No → don't guess
    return line.rstrip(".").strip()

def answer_question(question, plan, results):
    qtype = plan["question_type"]
    passages, evidence_text = passages_for(results, qtype)
    prompt = fill(ANSWER_PROMPT, passages=passages, question=question, instruction=INSTRUCTIONS[qtype])
    raw = call_llm(prompt, ANSWER_MODEL, "answer", max_tokens=64)
    return parse_answer(raw, qtype), evidence_text

## 12. Answer matching and confidence signals

**Matching (evaluation).** The gold answers in `train.json` are short: Yes/No, or a 1–3 word name. So most checks are exact, after normalising. Names also accept word-level containment ("Caesars" vs "Caesars Sportsbook") and pairs in any order. A few gold labels aren't Yes/No (e.g. "Consistent", "Agreement", about 2%). They're excluded from accuracy and counted, rather than guessed.

**Confidence signals** are measured, not self-reported (LLM self-confidence is poorly calibrated): the weakest fact's retrieval score, the average score, whether evidence came from the named outlet, whether a name answer actually appears in the evidence, the number of facts, and whether repair was needed.

In [ ]:
def norm(s):
    return " ".join(re.findall(r"[a-z0-9]+", str(s).lower()))

def strip_the(s):
    return re.sub(r"^the ", "", s)

def gold_yes_no(answer):
    n = norm(answer)
    return "yes" if n in ("yes", "true") else "no" if n in ("no", "false") else None

def name_set(s):
    return {strip_the(p.strip()) for p in re.split(r"\band\b|&|,", norm(s)) if p.strip()}

def names_match(pred, gold):
    p, g = strip_the(norm(pred)), strip_the(norm(gold))
    if not p:
        return False
    if p == g:
        return True
    shorter, longer = sorted([p, g], key=len)
    if len(shorter) >= 4 and f" {shorter} " in f" {longer} ":           # whole-word containment
        return True
    ps, gs = name_set(pred), name_set(gold)
    if len(gs) > 1 and len(ps) == len(gs):                               # "A and B" in any order
        return all(any(a == b or (min(len(a), len(b)) >= 4 and (a in b or b in a)) for b in ps) for a in gs)
    return False

JUDGE_PROMPT = '''TASK: JUDGE
Decide whether the model's answer refers to the same entity as the reference answer.
SAME: the same person, organisation or thing, even if written differently (abbreviation, nickname, with or without a qualifier).
DIFFERENT: a different entity, even if related or similar.
For pairs ("A and B"): SAME only if both entities match, in any order.

Question: <<question>>
Reference answer: <<gold>>
Model's answer: <<pred>>
Reply SAME or DIFFERENT only.'''

JUDGED = []                                                              # kept for the human calibration check

def is_correct(pred, q):
    qtype = q["question_type"]
    if qtype == "null_query":
        return pred == ABSTAIN
    if pred == ABSTAIN:
        return False
    if qtype in ("comparison_query", "temporal_query"):
        return norm(pred) == gold_yes_no(q["answer"])
    if names_match(pred, q["answer"]):
        return True
    if USE_LLM_JUDGE:
        verdict = call_llm(fill(JUDGE_PROMPT, question=q["query"], gold=q["answer"], pred=pred), JUDGE_MODEL, "judge", max_tokens=8)
        same = verdict.strip().upper().startswith("SAME")
        JUDGED.append({"qid": q["qid"], "question": q["query"], "gold": q["answer"], "pred": pred, "judge": "SAME" if same else "DIFFERENT"})
        return same
    return False

def scorable(q):
    # exclude the ~2% of Yes/No questions whose gold label isn't Yes/No
    return not (q["question_type"] in ("comparison_query", "temporal_query") and gold_yes_no(q["answer"]) is None)

STOP = {"the", "and", "of", "a", "an", "in", "on", "for", "to", "inc", "ltd"}

def grounded(answer, qtype, evidence_text):
    if qtype != "inference_query":
        return 1.0
    words = [w for w in norm(answer).split() if w not in STOP and len(w) > 2]
    if not words:
        return 0.0
    ev = norm(evidence_text)
    return sum(f" {w} " in f" {ev} " for w in words) / len(words)

FEATURE_NAMES = ["weakest_score", "mean_score", "share_from_named_outlet", "grounded", "n_facts", "repaired"]

def features(rec):
    tops = [r["best"] for r in rec["retrieval"]] or [0.0]
    named = [(r["level"] <= 2) if r["fact"]["source_key"] else True for r in rec["retrieval"]] or [False]
    return [min(tops), float(np.mean(tops)), float(np.mean(named)), rec["grounded"],
            len(rec["retrieval"]), float(rec["repaired"])]

## 13. The full pipeline for one question

`evaluation_mode=True` always runs retrieval and the answer call, even when the system would abstain, and records why. The **with vs without gating** comparison then comes from the same run. In live use (`evaluation_mode=False`) the pipeline stops as soon as it knows it will abstain, so it makes no wasted calls.

In [ ]:
def run_question(q, min_score, evaluation_mode=True):
    calls_before = sum(CALLS.values())
    question = q["query"]
    plan = plan_question(question)
    unknown_outlet = any(f["source_written"] and not f["source_known"] for f in plan["facts"])
    rec = {"qid": q["qid"], "query": question, "gold_type": q["question_type"], "gold_answer": q["answer"],
           "plan": plan, "pred_type": plan["question_type"], "unknown_outlet": unknown_outlet,
           "repaired": False, "retrieval": [], "raw_answer": ABSTAIN, "grounded": 0.0}

    if unknown_outlet and USE_UNKNOWN_OUTLET_EXIT and not evaluation_mode:
        rec["calls"] = sum(CALLS.values()) - calls_before
        return rec

    results = retrieve_all(plan, min_score)
    if n_strong(results) < len(results):
        new_plan = repair_plan(question, plan, results)                   # at most once
        rec["repaired"] = True
        if new_plan is not None:
            new_results = retrieve_all(new_plan, min_score)
            if n_strong(new_results) >= n_strong(results):
                plan, results = new_plan, new_results
    rec["plan"], rec["retrieval"] = plan, results
    rec["code_null"] = n_strong(results) < len(results)                  # a required fact has no strong evidence

    if evaluation_mode or not rec["code_null"]:
        answer, evidence_text = answer_question(question, plan, results)
        rec["raw_answer"] = answer
        rec["grounded"] = grounded(answer, plan["question_type"], evidence_text)
    rec["features"] = features(rec)
    rec["calls"] = sum(CALLS.values()) - calls_before
    return rec

def gate(rec, conf_model=None, conf_threshold=None):
    # final answer after all abstention rules
    if USE_UNKNOWN_OUTLET_EXIT and rec["unknown_outlet"]:
        return ABSTAIN, "outlet not in corpus"
    if rec.get("code_null", True):
        return ABSTAIN, "missing evidence"
    if rec["raw_answer"] == ABSTAIN:
        return ABSTAIN, "model abstained"
    if conf_model is not None:
        confidence = conf_model.predict_proba([rec["features"]])[0, 1]
        if confidence < conf_threshold:
            return ABSTAIN, "low confidence"
    return rec["raw_answer"], "answered"

def run_many(qs, min_score, label):
    out = []
    for i, q in enumerate(qs, 1):
        out.append(run_question(q, min_score))
        if i % 25 == 0 or i == len(qs):
            print(f"  {label}: {i}/{len(qs)} questions, API calls so far {dict(CALLS)}")
            save_cache()
    return out

## 14. Calibration on the validation split

**14a. Retrieval threshold `MIN_SCORE`.** Plan the validation questions, record each fact's best achievable score (unfiltered search, the loosest level), and sweep thresholds. A question is predicted unanswerable if its weakest fact falls below the threshold. Pick the threshold with the best balance between unanswerable questions caught and answerable questions wrongly rejected.

In [ ]:
val_plans = {}
val_weakest = []
for i, q in enumerate(val_qs, 1):
    plan = plan_question(q["query"])
    val_plans[q["qid"]] = plan
    qvecs = embed_texts([f["text"] for f in plan["facts"]])
    weakest = min(search(v, None, 1)[0][1] for v in qvecs)
    unknown = any(f["source_written"] and not f["source_known"] for f in plan["facts"])
    val_weakest.append({"is_null": q["question_type"] == "null_query", "weakest": weakest, "unknown": unknown})
    if i % 50 == 0:
        save_cache()
vw = pd.DataFrame(val_weakest)

rows = []
for t in np.round(np.arange(0.20, 0.86, 0.025), 3):
    pred_null = (vw["weakest"] < t) | (vw["unknown"] & USE_UNKNOWN_OUTLET_EXIT)
    tpr = (pred_null & vw["is_null"]).sum() / max(1, vw["is_null"].sum())             # nulls caught
    fpr = (pred_null & ~vw["is_null"]).sum() / max(1, (~vw["is_null"]).sum())         # answerable wrongly rejected
    rows.append({"threshold": t, "nulls_caught": round(tpr, 3), "answerable_rejected": round(fpr, 3),
                 "balanced": round((tpr + 1 - fpr) / 2, 3)})
sweep = pd.DataFrame(rows)
MIN_SCORE = float(sweep.loc[sweep["balanced"].idxmax(), "threshold"])
print(sweep.to_string(index=False))
print(f"\nMIN_SCORE = {MIN_SCORE}")
if USE_UNKNOWN_OUTLET_EXIT:
    print(f"unknown-outlet rule alone: flags {vw.loc[vw.is_null, 'unknown'].mean():.0%} of validation nulls "
          f"and {vw.loc[~vw.is_null, 'unknown'].mean():.0%} of answerable questions")

**14b. Confidence model and threshold.** Run the full pipeline on validation, fit a logistic regression that predicts whether an answer is correct from the measured signals, and sweep the confidence threshold. The chosen threshold maximises *correct − COST_OF_WRONG × wrong* over all validation questions (answering an unanswerable question counts as wrong). The whole risk–coverage table is printed, so the trade-off is visible rather than tuned to a target number.

In [ ]:
val_records = run_many(val_qs, MIN_SCORE, "validation")
q_by_id = {q["qid"]: q for q in questions}

train_rows = [r for r in val_records if not r.get("code_null", True) and r["raw_answer"] != ABSTAIN
              and not (USE_UNKNOWN_OUTLET_EXIT and r["unknown_outlet"]) and scorable(q_by_id[r["qid"]])]
X = np.array([r["features"] for r in train_rows])
y = np.array([is_correct(r["raw_answer"], q_by_id[r["qid"]]) for r in train_rows], dtype=int)
print(f"confidence model trained on {len(y)} answered validation questions, {y.mean():.0%} correct")

if len(set(y)) == 2:
    conf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y)
    coefs = conf_model[-1].coef_[0]
    print(pd.Series(coefs, index=FEATURE_NAMES).round(2).to_string())
else:
    conf_model = None
    print("only one class in the validation answers: confidence gating disabled")

def outcome_table(records, conf_model, thresholds):
    rows = []
    for t in thresholds:
        answered = correct = 0
        for r in records:
            q = q_by_id[r["qid"]]
            if not scorable(q):
                continue
            final, _ = gate(r, conf_model, t)
            if final != ABSTAIN:
                answered += 1
                correct += is_correct(final, q)
        total = sum(scorable(q_by_id[r["qid"]]) for r in records)
        rows.append({"threshold": t, "coverage": round(answered / max(1, total), 3),
                     "accuracy_on_answered": round(correct / max(1, answered), 3),
                     "utility": correct - COST_OF_WRONG * (answered - correct)})
    return pd.DataFrame(rows)

if conf_model is not None:
    val_table = outcome_table(val_records, conf_model, np.round(np.arange(0.0, 0.96, 0.05), 2))
    CONF_THRESHOLD = float(val_table.loc[val_table["utility"].idxmax(), "threshold"])
    print(val_table.to_string(index=False))
else:
    CONF_THRESHOLD = 0.0
print(f"\nCONF_THRESHOLD = {CONF_THRESHOLD}")

## 15. Test run and evaluation

Everything chosen above (examples, `MIN_SCORE`, the confidence model and its threshold) is now fixed. The test split is run **once**.

In [ ]:
test_records = run_many(test_qs, MIN_SCORE, "test")
for r in test_records:
    r["final_answer"], r["reason"] = gate(r, conf_model, CONF_THRESHOLD)
save_cache()
print("API calls per question (evaluation mode):", pd.Series([r["calls"] for r in test_records]).describe()[["mean", "max"]].round(2).to_dict())

### 15a. Question-type misclassification: bare prompt vs planner prompt

Same answerable test questions, two prompts. The bare prompt gives only the three label names; the planner prompt adds definitions, the timing rule, examples and validated JSON. McNemar's test uses only the questions where the two disagree.

In [ ]:
BARE_PROMPT = '''TASK: CLASSIFY
Classify the question as inference_query, comparison_query or temporal_query.
Question: <<question>>'''

def bare_type(question):
    text = call_llm(fill(BARE_PROMPT, question=question), PLANNER_MODEL, "bare_classify", max_tokens=16).lower()
    for t in VALID_TYPES:
        if t in text or t.replace("_query", "") in text:
            return t
    return None                                                      # unusable reply counts as an error

answerable = [r for r in test_records if r["gold_type"] != "null_query"]
bare_ok = np.array([bare_type(r["query"]) == r["gold_type"] for r in answerable])
plan_ok = np.array([r["pred_type"] == r["gold_type"] for r in answerable])
save_cache()

err_bare, err_plan = 1 - bare_ok.mean(), 1 - plan_ok.mean()
b = int((bare_ok & ~plan_ok).sum())                                  # bare right, planner wrong
c = int((~bare_ok & plan_ok).sum())                                  # bare wrong, planner right
p_value = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
print(f"error rate: bare {err_bare:.1%} → planner {err_plan:.1%} "
      f"(relative reduction {(err_bare - err_plan) / max(err_bare, 1e-9):.0%}, absolute {100 * (err_bare - err_plan):.1f} points)")
print(f"McNemar: planner fixed {c}, broke {b}, p = {p_value:.3g}  (n = {len(answerable)})")
print(pd.crosstab(pd.Series([r["gold_type"] for r in answerable], name="gold"),
                  pd.Series([r["pred_type"] for r in answerable], name="planner")))

### 15b. Retrieval relevance against the gold evidence articles

Judged automatically: a retrieved chunk is relevant if it comes from one of the question's gold evidence articles (`evidence_list` URLs). No human labelling.

In [ ]:
relevant = retrieved = 0
fact_hits, fact_total, recalls = 0, 0, []
for r in answerable:
    gold_urls = {e["url"] for e in q_by_id[r["qid"]]["evidence_list"]}
    got = set()
    for fr in r["retrieval"]:
        urls = [h["url"] for h in fr["hits"][:TOP_K]]
        retrieved += len(urls)
        relevant += sum(u in gold_urls for u in urls)
        fact_total += 1
        fact_hits += any(u in gold_urls for u in urls)
        got |= set(urls)
    if gold_urls:
        recalls.append(len(gold_urls & got) / len(gold_urls))

print(f"precision@{TOP_K}: {relevant / max(1, retrieved):.1%}  ({relevant}/{retrieved} chunks from a gold article)")
print(f"facts with at least one gold chunk: {fact_hits / max(1, fact_total):.1%}")
print(f"evidence recall per question: {np.mean(recalls):.1%}")

### 15c. Answers: accuracy on answered questions, coverage, hallucinations, baselines

In [ ]:
scored = [r for r in test_records if scorable(q_by_id[r["qid"]])]
excluded = len(test_records) - len(scored)
answered = [r for r in scored if r["final_answer"] != ABSTAIN]
correct = [r for r in answered if is_correct(r["final_answer"], q_by_id[r["qid"]])]

nulls = [r for r in test_records if r["gold_type"] == "null_query"]
false_ungated = sum(r["raw_answer"] != ABSTAIN for r in nulls) / max(1, len(nulls))
false_gated = sum(r["final_answer"] != ABSTAIN for r in nulls) / max(1, len(nulls))
ungated_answered = [r for r in scored if r["raw_answer"] != ABSTAIN]
ungated_correct = sum(is_correct(r["raw_answer"], q_by_id[r["qid"]]) for r in ungated_answered)

print(f"accuracy on answered questions: {len(correct) / max(1, len(answered)):.1%} "
      f"at coverage {len(answered) / max(1, len(scored)):.1%} ({len(answered)}/{len(scored)} answered)")
print(f"without gating: accuracy {ungated_correct / max(1, len(ungated_answered)):.1%} "
      f"at coverage {len(ungated_answered) / max(1, len(scored)):.1%}")
print(f"false-answer rate on unanswerable questions: {false_ungated:.1%} without gating → {false_gated:.1%} with gating")
print(f"(excluded {excluded} questions whose gold label isn't Yes/No)")
print("abstention reasons:", Counter(r["reason"] for r in test_records))

val_majority = {t: Counter(norm(q["answer"]) for q in val_qs if q["question_type"] == t).most_common(1)[0][0]
                for t in VALID_TYPES}
rows = []
for t in VALID_TYPES:
    rs = [r for r in scored if r["gold_type"] == t]
    ans = [r for r in rs if r["final_answer"] != ABSTAIN]
    rows.append({"type": t, "questions": len(rs), "coverage": round(len(ans) / max(1, len(rs)), 3),
                 "accuracy_on_answered": round(sum(is_correct(r["final_answer"], q_by_id[r["qid"]]) for r in ans) / max(1, len(ans)), 3),
                 "majority_baseline": round(sum(norm(r["gold_answer"]) == val_majority[t] for r in rs) / max(1, len(rs)), 3),
                 "baseline_answer": val_majority[t]})
print(pd.DataFrame(rows).to_string(index=False))

if conf_model is not None:
    print("\nrisk–coverage on test:")
    print(outcome_table(test_records, conf_model, [0.0, 0.3, 0.5, CONF_THRESHOLD, 0.8, 0.9]).to_string(index=False))

### 15d. Checking the LLM judge against a human

The judge only sees name answers that code couldn't match. Label the exported sample yourself (`human` column: SAME / DIFFERENT), save it as `judge_human_labels.csv` in `WORK_DIR`, and re-run this cell to measure agreement. Only trust the judge if agreement is high.

In [ ]:
judged = pd.DataFrame(JUDGED).drop_duplicates("qid")
sample_path = os.path.join(WORK_DIR, "judge_sample_for_human.csv")
labels_path = os.path.join(WORK_DIR, "judge_human_labels.csv")
if len(judged):
    judged.head(100).assign(human="").to_csv(sample_path, index=False)
    print(f"{len(judged)} name answers went to the judge; {min(100, len(judged))} exported to {sample_path}")
    if os.path.exists(labels_path):
        h = pd.read_csv(labels_path).dropna(subset=["human"])
        h = h[h["human"].str.upper().isin(["SAME", "DIFFERENT"])]
        if len(h):
            agree = (h["human"].str.upper() == h["judge"]).mean()
            kappa = cohen_kappa_score(h["human"].str.upper(), h["judge"])
            print(f"judge vs human on {len(h)} cases: agreement {agree:.0%}, Cohen's kappa {kappa:.2f}")
else:
    print("no answers needed the judge")

## 16. Save predictions in the dataset's format, and summary

In [ ]:
def to_output(r):
    evidence = []
    if r["final_answer"] != ABSTAIN:
        for fr in r["retrieval"]:
            if fr["hits"]:
                h = fr["hits"][0]
                evidence.append({"title": h["title"], "source": h["source"], "published_at": h["published_at"],
                                 "url": h["url"], "fact": h["text"][:300]})
    return {"query": r["query"], "answer": r["final_answer"],
            "question_type": "null_query" if r["final_answer"] == ABSTAIN else r["pred_type"],
            "evidence_list": evidence}

out_path = os.path.join(WORK_DIR, "test_predictions.json")
with open(out_path, "w") as f:
    json.dump([to_output(r) for r in test_records], f, indent=2)
save_cache()

print(f"saved {len(test_records)} predictions to {out_path}\n")
print("SUMMARY (test split, measured in this run)")
print(f"- question-type error: {err_bare:.1%} → {err_plan:.1%} ({(err_bare - err_plan) / max(err_bare, 1e-9):.0%} relative reduction, McNemar p = {p_value:.3g})")
print(f"- retrieval precision@{TOP_K} vs gold evidence: {relevant / max(1, retrieved):.1%}")
print(f"- accuracy on answered questions: {len(correct) / max(1, len(answered)):.1%} at {len(answered) / max(1, len(scored)):.1%} coverage")
print(f"- false answers on unanswerable questions: {false_ungated:.1%} → {false_gated:.1%}")
print(f"- thresholds chosen on validation: MIN_SCORE = {MIN_SCORE}, CONF_THRESHOLD = {CONF_THRESHOLD}")
print(f"- API calls made in this session: {dict(CALLS)}")

## Notes

- **Workflow, not a free-roaming agent.** Code fixes the order of steps. The LLM decides the plan and the repair, and those decisions change which searches run, with which filters. That gives at most three LLM calls per question in live use (plus a JSON retry when a reply is malformed).
- **Dataset-specific rule.** `USE_UNKNOWN_OUTLET_EXIT` works because many unanswerable questions in this benchmark cite outlets the corpus doesn't have. Section 14a prints how much it contributes. Turn it off to see the system without it.
- **Evaluation mode costs more** than live use, because it answers even questions it will abstain on. That's what makes the with/without-gating comparison possible.
- **Next improvements:** BM25 alongside embeddings for exact names and numbers (hybrid search); a cross-encoder reranker; per-claim checks for comparison questions; agreement across several sampled answers as an extra confidence signal.